In [1]:
from pathlib import Path
import pandas as pd
import json
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

METRICA_ROOT = PROJECT_ROOT / "external" / "metrica"
METRICA_DATA = METRICA_ROOT / "data"

print("Metrica root:", METRICA_ROOT)
print("Exists:", METRICA_ROOT.exists())
print("Data exists:", METRICA_DATA.exists())

for item in METRICA_DATA.iterdir():
    print(item.name)

Metrica root: c:\Users\amsch\. Personal Work\Projects\Soccer TIPS\external\metrica
Exists: True
Data exists: True
Sample_Game_1
Sample_Game_2
Sample_Game_3


In [2]:
expected_metrica_files = {
    "Sample_Game_1": [
        "Sample_Game_1_RawEventsData.csv",
        "Sample_Game_1_RawTrackingData_Away_Team.csv",
        "Sample_Game_1_RawTrackingData_Home_Team.csv",
    ],
    "Sample_Game_2": [
        "Sample_Game_2_RawEventsData.csv",
        "Sample_Game_2_RawTrackingData_Away_Team.csv",
        "Sample_Game_2_RawTrackingData_Home_Team.csv",
    ],
    "Sample_Game_3": [
        "Sample_Game_3_events.json",
        "Sample_Game_3_metadata.xml",
        "Sample_Game_3_tracking.txt",
    ],
}

rows = []

for game, files in expected_metrica_files.items():
    game_folder = METRICA_DATA / game
    
    row = {
        "game": game,
        "folder_exists": game_folder.exists()
    }
    
    for file in files:
        row[file] = (game_folder / file).exists()
    
    rows.append(row)

metrica_file_check = pd.DataFrame(rows)
metrica_file_check

,game,folder_exists,Sample_Game_1_RawEventsData.csv,Sample_Game_1_RawTrackingData_Away_Team.csv,Sample_Game_1_RawTrackingData_Home_Team.csv,Sample_Game_2_RawEventsData.csv,Sample_Game_2_RawTrackingData_Away_Team.csv,Sample_Game_2_RawTrackingData_Home_Team.csv,Sample_Game_3_events.json,Sample_Game_3_metadata.xml,Sample_Game_3_tracking.txt
0,Sample_Game_1,True,True,True,True,NaN,NaN,NaN,NaN,NaN,NaN
1,Sample_Game_2,True,NaN,NaN,NaN,True,True,True,NaN,NaN,NaN
2,Sample_Game_3,True,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True


In [3]:
metrica_file_check.all()

game                                           True
folder_exists                                  True
Sample_Game_1_RawEventsData.csv                True
Sample_Game_1_RawTrackingData_Away_Team.csv    True
Sample_Game_1_RawTrackingData_Home_Team.csv    True
Sample_Game_2_RawEventsData.csv                True
Sample_Game_2_RawTrackingData_Away_Team.csv    True
Sample_Game_2_RawTrackingData_Home_Team.csv    True
Sample_Game_3_events.json                      True
Sample_Game_3_metadata.xml                     True
Sample_Game_3_tracking.txt                     True
dtype: bool

In [4]:
# ---------- Sample Game 1 + 2 CSV Validation ----------

csv_games = ["Sample_Game_1", "Sample_Game_2"]

csv_validation_rows = []

for game in csv_games:
    game_folder = METRICA_DATA / game
    
    events_file = game_folder / f"{game}_RawEventsData.csv"
    home_tracking_file = game_folder / f"{game}_RawTrackingData_Home_Team.csv"
    away_tracking_file = game_folder / f"{game}_RawTrackingData_Away_Team.csv"
    
    events_df = pd.read_csv(events_file)
    home_df = pd.read_csv(home_tracking_file, header=None)
    away_df = pd.read_csv(away_tracking_file, header=None)
    
    csv_validation_rows.append({
        "game": game,
        "events_shape": events_df.shape,
        "home_tracking_shape_raw": home_df.shape,
        "away_tracking_shape_raw": away_df.shape,
        "events_columns": list(events_df.columns),
        "home_first_row_preview": list(home_df.iloc[0, :8]),
        "away_first_row_preview": list(away_df.iloc[0, :8]),
    })

csv_validation_df = pd.DataFrame(csv_validation_rows)
csv_validation_df

C:\Users\amsch\AppData\Local\Temp\ipykernel_9268\1810856035.py:15: DtypeWarning: Columns (0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 7, 6: 9, 7: 11, 8: 13, 9: 15, 10: 17, 11: 19, 12: 21, 13: 23, 14: 25, 15: 27, 16: 29, 17: 31) have mixed types. Specify dtype option on import or set low_memory=False.
  home_df = pd.read_csv(home_tracking_file, header=None)
C:\Users\amsch\AppData\Local\Temp\ipykernel_9268\1810856035.py:16: DtypeWarning: Columns (0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 7, 6: 9, 7: 11, 8: 13, 9: 15, 10: 17, 11: 19, 12: 21, 13: 23, 14: 25, 15: 27, 16: 29, 17: 31) have mixed types. Specify dtype option on import or set low_memory=False.
  away_df = pd.read_csv(away_tracking_file, header=None)
C:\Users\amsch\AppData\Local\Temp\ipykernel_9268\1810856035.py:16: DtypeWarning: Columns (0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 7, 6: 9, 7: 11, 8: 13, 9: 15, 10: 17, 11: 19, 12: 21, 13: 23, 14: 25, 15: 27) have mixed types. Specify dtype option on import or set low_memory=False.
  away_df = pd.read_csv(away_

,game,events_shape,home_tracking_shape_raw,away_tracking_shape_raw,events_columns,home_first_row_preview,away_first_row_preview
0,Sample_Game_1,"(1745, 14)","(145009, 33)","(145009, 33)","[Team, Type, Subtype, Period, Start Frame, Sta...","[nan, nan, nan, Home, nan, Home, nan, Home]","[nan, nan, nan, Away, nan, Away, nan, Away]"
1,Sample_Game_2,"(1935, 14)","(141159, 33)","(141159, 29)","[Team, Type, Subtype, Period, Start Frame, Sta...","[nan, nan, nan, Home, nan, Home, nan, Home]","[nan, nan, nan, Away, nan, Away, nan, Away]"


In [5]:
for row in csv_validation_rows:
    print("\n", row["game"])
    print("events_shape:", row["events_shape"])
    print("home_tracking_shape_raw:", row["home_tracking_shape_raw"])
    print("away_tracking_shape_raw:", row["away_tracking_shape_raw"])
    print("events_columns:", row["events_columns"])
    print("home_first_row_preview:", row["home_first_row_preview"])
    print("away_first_row_preview:", row["away_first_row_preview"])


 Sample_Game_1
events_shape: (1745, 14)
home_tracking_shape_raw: (145009, 33)
away_tracking_shape_raw: (145009, 33)
events_columns: ['Team', 'Type', 'Subtype', 'Period', 'Start Frame', 'Start Time [s]', 'End Frame', 'End Time [s]', 'From', 'To', 'Start X', 'Start Y', 'End X', 'End Y']
home_first_row_preview: [nan, nan, nan, 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home']
away_first_row_preview: [nan, nan, nan, 'Away', np.float64(nan), 'Away', np.float64(nan), 'Away']

 Sample_Game_2
events_shape: (1935, 14)
home_tracking_shape_raw: (141159, 33)
away_tracking_shape_raw: (141159, 29)
events_columns: ['Team', 'Type', 'Subtype', 'Period', 'Start Frame', 'Start Time [s]', 'End Frame', 'End Time [s]', 'From', 'To', 'Start X', 'Start Y', 'End X', 'End Y']
home_first_row_preview: [nan, nan, nan, 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home']
away_first_row_preview: [nan, nan, nan, 'Away', np.float64(nan), 'Away', np.float64(nan), 'Away']


In [6]:
# ---------- Inspect Header Structure ----------

game = "Sample_Game_1"

game_folder = METRICA_DATA / game

home_tracking_file = game_folder / f"{game}_RawTrackingData_Home_Team.csv"

raw_tracking = pd.read_csv(home_tracking_file, header=None)

for i in range(6):
    print(f"\nROW {i}")
    print(raw_tracking.iloc[i, :15].tolist())


ROW 0
[nan, nan, nan, 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home', np.float64(nan), 'Home', np.float64(nan)]

ROW 1
[nan, nan, nan, '11', np.float64(nan), '1', np.float64(nan), '2', np.float64(nan), '3', np.float64(nan), '4', np.float64(nan), '5', np.float64(nan)]

ROW 2
['Period', 'Frame', 'Time [s]', 'Player11', np.float64(nan), 'Player1', np.float64(nan), 'Player2', np.float64(nan), 'Player3', np.float64(nan), 'Player4', np.float64(nan), 'Player5', np.float64(nan)]

ROW 3
['1', '1', '0.04', '0.00082', np.float64(0.48238), '0.32648', np.float64(0.65322), '0.33701', np.float64(0.48863), '0.30927', np.float64(0.35529), '0.32137', np.float64(0.21262), '0.41094', np.float64(0.72589)]

ROW 4
['1', '2', '0.08', '0.00096', np.float64(0.48238), '0.32648', np.float64(0.65322), '0.33701', np.float64(0.48863), '0.30927', np.float64(0.35529), '0.32137', np.float64(0.21262), '0.41094', np.float64(0.72589)]

ROW 5
['1', '3', '0.12', '

C:\Users\amsch\AppData\Local\Temp\ipykernel_9268\633697632.py:9: DtypeWarning: Columns (0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 7, 6: 9, 7: 11, 8: 13, 9: 15, 10: 17, 11: 19, 12: 21, 13: 23, 14: 25, 15: 27, 16: 29, 17: 31) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_tracking = pd.read_csv(home_tracking_file, header=None)


In [7]:
game = "Sample_Game_1"
game_folder = METRICA_DATA / game

home_tracking_file = game_folder / f"{game}_RawTrackingData_Home_Team.csv"
away_tracking_file = game_folder / f"{game}_RawTrackingData_Away_Team.csv"
events_file = game_folder / f"{game}_RawEventsData.csv"

home_tracking = pd.read_csv(home_tracking_file, skiprows=3)
away_tracking = pd.read_csv(away_tracking_file, skiprows=3)
events = pd.read_csv(events_file)

print("home_tracking:", home_tracking.shape)
print("away_tracking:", away_tracking.shape)
print("events:", events.shape)

print("\nHome columns preview:")
print(home_tracking.columns[:12])

print("\nAway columns preview:")
print(away_tracking.columns[:12])

print("\nEvents columns:")
print(events.columns.tolist())

home_tracking: (145005, 33)
away_tracking: (145005, 33)
events: (1745, 14)

Home columns preview:
Index(['1', '1.1', '0.04', '0.00082', '0.48238', '0.32648', '0.65322',
       '0.33701', '0.48863', '0.30927', '0.35529', '0.32137'],
      dtype='str')

Away columns preview:
Index(['1', '1.1', '0.04', '0.90509', '0.47462', '0.58393', '0.20794',
       '0.67658', '0.4671', '0.6731', '0.76476', '0.40783'],
      dtype='str')

Events columns:
['Team', 'Type', 'Subtype', 'Period', 'Start Frame', 'Start Time [s]', 'End Frame', 'End Time [s]', 'From', 'To', 'Start X', 'Start Y', 'End X', 'End Y']


In [8]:
game = "Sample_Game_1"
game_folder = METRICA_DATA / game

home_tracking_file = game_folder / f"{game}_RawTrackingData_Home_Team.csv"
away_tracking_file = game_folder / f"{game}_RawTrackingData_Away_Team.csv"

home_tracking = pd.read_csv(home_tracking_file, skiprows=2)
away_tracking = pd.read_csv(away_tracking_file, skiprows=2)

print("home_tracking:", home_tracking.shape)
print("away_tracking:", away_tracking.shape)

print("\nHome columns:")
print(home_tracking.columns.tolist()[:20])

print("\nAway columns:")
print(away_tracking.columns.tolist()[:20])

display(home_tracking.head())

home_tracking: (145006, 33)
away_tracking: (145006, 33)

Home columns:
['Period', 'Frame', 'Time [s]', 'Player11', 'Unnamed: 4', 'Player1', 'Unnamed: 6', 'Player2', 'Unnamed: 8', 'Player3', 'Unnamed: 10', 'Player4', 'Unnamed: 12', 'Player5', 'Unnamed: 14', 'Player6', 'Unnamed: 16', 'Player7', 'Unnamed: 18', 'Player8']

Away columns:
['Period', 'Frame', 'Time [s]', 'Player25', 'Unnamed: 4', 'Player15', 'Unnamed: 6', 'Player16', 'Unnamed: 8', 'Player17', 'Unnamed: 10', 'Player18', 'Unnamed: 12', 'Player19', 'Unnamed: 14', 'Player20', 'Unnamed: 16', 'Player21', 'Unnamed: 18', 'Player22']


,Period,Frame,Time [s],Player11,Unnamed: 4,Player1,Unnamed: 6,Player2,Unnamed: 8,Player3,...,Player10,Unnamed: 24,Player12,Unnamed: 26,Player13,Unnamed: 28,Player14,Unnamed: 30,Ball,Unnamed: 32
0,1,1,0.04,0.00082,0.48238,0.32648,0.65322,0.33701,0.48863,0.30927,...,0.55243,0.43269,NaN,NaN,NaN,NaN,NaN,NaN,0.45472,0.38709
1,1,2,0.08,0.00096,0.48238,0.32648,0.65322,0.33701,0.48863,0.30927,...,0.55243,0.43269,NaN,NaN,NaN,NaN,NaN,NaN,0.49645,0.40656
2,1,3,0.12,0.00114,0.48238,0.32648,0.65322,0.33701,0.48863,0.30927,...,0.55243,0.43269,NaN,NaN,NaN,NaN,NaN,NaN,0.53716,0.42556
3,1,4,0.16,0.00121,0.48238,0.32622,0.65317,0.33687,0.48988,0.30944,...,0.55236,0.43313,NaN,NaN,NaN,NaN,NaN,NaN,0.55346,0.42231
4,1,5,0.20,0.00129,0.48238,0.32597,0.65269,0.33664,0.49018,0.30948,...,0.55202,0.43311,NaN,NaN,NaN,NaN,NaN,NaN,0.55512,0.40570


In [9]:
validation_summary = {
    "home_missing_total": home_tracking.isna().sum().sum(),
    "away_missing_total": away_tracking.isna().sum().sum(),
    
    "home_min_frame": home_tracking["Frame"].min(),
    "home_max_frame": home_tracking["Frame"].max(),
    
    "away_min_frame": away_tracking["Frame"].min(),
    "away_max_frame": away_tracking["Frame"].max(),
    
    "home_periods": home_tracking["Period"].unique().tolist(),
    "away_periods": away_tracking["Period"].unique().tolist(),
}

validation_summary

{'home_missing_total': np.int64(983540),
 'away_missing_total': np.int64(983540),
 'home_min_frame': np.int64(1),
 'home_max_frame': np.int64(145006),
 'away_min_frame': np.int64(1),
 'away_max_frame': np.int64(145006),
 'home_periods': [1, 2],
 'away_periods': [1, 2]}

In [10]:
# ---------- Frame / Time / Event Alignment Validation ----------

events_file = game_folder / f"{game}_RawEventsData.csv"
events = pd.read_csv(events_file)

frame_time_check = {
    "home_frame_unique_count": home_tracking["Frame"].nunique(),
    "home_frame_row_count": len(home_tracking),
    "home_frame_duplicates": home_tracking["Frame"].duplicated().sum(),
    "home_frame_gaps": int((home_tracking["Frame"].diff().dropna() != 1).sum()),
    
    "away_frame_unique_count": away_tracking["Frame"].nunique(),
    "away_frame_row_count": len(away_tracking),
    "away_frame_duplicates": away_tracking["Frame"].duplicated().sum(),
    "away_frame_gaps": int((away_tracking["Frame"].diff().dropna() != 1).sum()),
    
    "home_time_min": home_tracking["Time [s]"].min(),
    "home_time_max": home_tracking["Time [s]"].max(),
    "away_time_min": away_tracking["Time [s]"].min(),
    "away_time_max": away_tracking["Time [s]"].max(),
    
    "estimated_fps_from_first_step": round(1 / home_tracking["Time [s]"].diff().dropna().mode().iloc[0], 2),
    
    "event_min_start_frame": events["Start Frame"].min(),
    "event_max_end_frame": events["End Frame"].max(),
    "events_before_tracking": int((events["Start Frame"] < home_tracking["Frame"].min()).sum()),
    "events_after_tracking": int((events["End Frame"] > home_tracking["Frame"].max()).sum()),
    "invalid_event_ranges": int((events["End Frame"] < events["Start Frame"]).sum()),
}

frame_time_check

{'home_frame_unique_count': 145006,
 'home_frame_row_count': 145006,
 'home_frame_duplicates': np.int64(0),
 'home_frame_gaps': 0,
 'away_frame_unique_count': 145006,
 'away_frame_row_count': 145006,
 'away_frame_duplicates': np.int64(0),
 'away_frame_gaps': 0,
 'home_time_min': np.float64(0.04),
 'home_time_max': np.float64(5800.24),
 'away_time_min': np.float64(0.04),
 'away_time_max': np.float64(5800.24),
 'estimated_fps_from_first_step': np.float64(25.0),
 'event_min_start_frame': np.int64(1),
 'event_max_end_frame': np.int64(143630),
 'events_before_tracking': 0,
 'events_after_tracking': 0,
 'invalid_event_ranges': 1}

In [11]:
events["Type"].value_counts()

Type
PASS              799
RECOVERY          278
BALL LOST         257
CHALLENGE         233
SET PIECE          77
BALL OUT           51
SHOT               24
FAULT RECEIVED     22
CARD                4
Name: count, dtype: int64

In [12]:
events[
    events["End Frame"] < events["Start Frame"]
]

,Team,Type,Subtype,Period,Start Frame,Start Time [s],End Frame,End Time [s],From,To,Start X,Start Y,End X,End Y
0,Away,SET PIECE,KICK OFF,1,1,0.04,0,0.0,Player19,NaN,NaN,NaN,NaN,NaN


In [13]:
# ---------- Coordinate + Missingness Validation ----------

coordinate_summary = {
    "home_min_value": float(home_tracking.iloc[:, 3:].min().min()),
    "home_max_value": float(home_tracking.iloc[:, 3:].max().max()),
    
    "away_min_value": float(away_tracking.iloc[:, 3:].min().min()),
    "away_max_value": float(away_tracking.iloc[:, 3:].max().max()),
    
    "home_total_missing": int(home_tracking.iloc[:, 3:].isna().sum().sum()),
    "away_total_missing": int(away_tracking.iloc[:, 3:].isna().sum().sum()),
}

coordinate_summary

{'home_min_value': -0.05,
 'home_max_value': 1.05996,
 'away_min_value': -0.05,
 'away_max_value': 1.05996,
 'home_total_missing': 983540,
 'away_total_missing': 983540}

In [14]:
# ---------- Inspect Player Column Pairing ----------

print(home_tracking.columns.tolist())

['Period', 'Frame', 'Time [s]', 'Player11', 'Unnamed: 4', 'Player1', 'Unnamed: 6', 'Player2', 'Unnamed: 8', 'Player3', 'Unnamed: 10', 'Player4', 'Unnamed: 12', 'Player5', 'Unnamed: 14', 'Player6', 'Unnamed: 16', 'Player7', 'Unnamed: 18', 'Player8', 'Unnamed: 20', 'Player9', 'Unnamed: 22', 'Player10', 'Unnamed: 24', 'Player12', 'Unnamed: 26', 'Player13', 'Unnamed: 28', 'Player14', 'Unnamed: 30', 'Ball', 'Unnamed: 32']


In [15]:
# ---------- Count Fully Missing Player Columns ----------

missing_by_column = home_tracking.isna().sum().sort_values(ascending=False)

missing_by_column.head(15)

Unnamed: 30    121391
Player14       121391
Unnamed: 28    110848
Player13       110848
Unnamed: 6      98300
Player1         98300
Ball            56755
Unnamed: 32     56755
Player12        46705
Unnamed: 26     46705
Unnamed: 16     34157
Player6         34157
Player10        23614
Unnamed: 24     23614
Period              0
dtype: int64